# Catalyst Optimizer Deep Dive

This notebook focuses specifically on Apache Spark's Catalyst optimizer internals.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("CatalystOptimizerDemo") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.cbo.enabled", "true") \
    .getOrCreate()

## 1. Query Plan Phases

In [ ]:
# Load data
sales_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/sales_data.csv")
customers_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/customers.csv")

# Register as temp views
sales_df.createOrReplaceTempView("sales")
customers_df.createOrReplaceTempView("customers")

# Complex query to demonstrate optimization phases
query = """
SELECT 
    c.name,
    c.city,
    SUM(s.amount) as total_spent,
    COUNT(s.transaction_id) as transaction_count,
    AVG(s.amount) as avg_transaction
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
WHERE s.amount > 100 
    AND c.age > 25
    AND s.region IN ('North', 'South')
GROUP BY c.name, c.city
HAVING SUM(s.amount) > 200
ORDER BY total_spent DESC
"""

result_df = spark.sql(query)

print("=== LOGICAL PLAN (Unresolved) ===")
# This shows the initial parsed plan
print("Initial logical plan after parsing SQL")

print("\n=== LOGICAL PLAN (Analyzed) ===")
# This shows the plan after analysis (column resolution, type checking)
result_df.explain("simple")

print("\n=== LOGICAL PLAN (Optimized) ===")
# This shows the plan after Catalyst rule-based optimization
result_df.explain("extended")

print("\n=== PHYSICAL PLAN ===")
# This shows the final physical execution plan
result_df.explain("formatted")

## 2. Rule-Based Optimizations

In [ ]:
print("=== PREDICATE PUSHDOWN DEMONSTRATION ===")

# Query that benefits from predicate pushdown
pushdown_query = """
SELECT s.transaction_id, s.amount, c.name
FROM (
    SELECT * FROM sales WHERE amount > 500
) s
JOIN (
    SELECT * FROM customers WHERE age > 30
) c ON s.customer_id = c.customer_id
WHERE s.region = 'North'
"""

pushdown_df = spark.sql(pushdown_query)
pushdown_df.explain(True)

print("\n=== COLUMN PRUNING DEMONSTRATION ===")

# Only select needed columns
pruning_df = sales_df.join(customers_df, "customer_id") \
    .select("transaction_id", "amount", "name") \
    .filter(col("amount") > 100)

print("Column pruning optimization:")
pruning_df.explain(True)

print("\n=== CONSTANT FOLDING DEMONSTRATION ===")

# Expressions that can be folded at compile time
constant_folding = sales_df.select(
    col("transaction_id"),
    col("amount"),
    (lit(10) + lit(5)).alias("constant_expr"),
    (col("amount") * (lit(2) + lit(3))).alias("partial_folding")
)

print("Constant folding optimization:")
constant_folding.explain(True)

## 3. Cost-Based Optimization (CBO)

In [ ]:
print("=== COST-BASED OPTIMIZATION ===")

# Enable CBO and generate statistics
spark.conf.set("spark.sql.cbo.enabled", "true")
spark.conf.set("spark.sql.cbo.joinReorder.enabled", "true")

# Analyze tables to generate statistics
spark.sql("ANALYZE TABLE sales COMPUTE STATISTICS")
spark.sql("ANALYZE TABLE customers COMPUTE STATISTICS")
spark.sql("ANALYZE TABLE sales COMPUTE STATISTICS FOR COLUMNS customer_id, amount, region")
spark.sql("ANALYZE TABLE customers COMPUTE STATISTICS FOR COLUMNS customer_id, age, city")

# Query that benefits from CBO
cbo_query = """
SELECT c.name, s.amount, s.region
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
WHERE s.amount > 300 AND c.age BETWEEN 25 AND 40
"""

cbo_df = spark.sql(cbo_query)
print("CBO-optimized query plan:")
cbo_df.explain("cost")

# Show table statistics
print("\n=== TABLE STATISTICS ===")
spark.sql("DESCRIBE EXTENDED sales").filter(
    col("col_name").contains("Statistics")
).show(truncate=False)

spark.sql("DESCRIBE EXTENDED customers").filter(
    col("col_name").contains("Statistics")
).show(truncate=False)

## 4. Join Optimization Strategies

In [ ]:
print("=== JOIN OPTIMIZATION STRATEGIES ===")

# 1. Broadcast Hash Join
print("1. Broadcast Hash Join:")
broadcast_join = sales_df.join(broadcast(customers_df), "customer_id")
broadcast_join.explain()

# 2. Sort-Merge Join (disable broadcast)
print("\n2. Sort-Merge Join:")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
sort_merge_join = sales_df.join(customers_df, "customer_id")
sort_merge_join.explain()

# 3. Bucket Join (simulated)
print("\n3. Bucket Join Simulation:")
# Pre-partition both DataFrames on join key
bucketed_sales = sales_df.repartition(4, "customer_id")
bucketed_customers = customers_df.repartition(4, "customer_id")

bucket_join = bucketed_sales.join(bucketed_customers, "customer_id")
bucket_join.explain()

# Reset broadcast threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

# 4. Join Reordering with CBO
print("\n4. Join Reordering:")
products_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/products.csv")
products_df.createOrReplaceTempView("products")

# Multi-table join that can be reordered
multi_join_query = """
SELECT s.transaction_id, c.name, p.name as product_name
FROM sales s
JOIN products p ON s.product_id = p.product_id
JOIN customers c ON s.customer_id = c.customer_id
WHERE s.amount > 200
"""

multi_join_df = spark.sql(multi_join_query)
print("Multi-table join with reordering:")
multi_join_df.explain()

## 5. Advanced Catalyst Features

In [ ]:
print("=== ADVANCED CATALYST FEATURES ===")

# 1. Subquery optimization
print("1. Subquery Optimization:")
subquery_sql = """
SELECT c.name, c.city
FROM customers c
WHERE c.customer_id IN (
    SELECT s.customer_id 
    FROM sales s 
    WHERE s.amount > 500
)
"""

subquery_df = spark.sql(subquery_sql)
subquery_df.explain()

# 2. Window function optimization
print("\n2. Window Function Optimization:")
window_sql = """
SELECT 
    customer_id,
    amount,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) as rank,
    SUM(amount) OVER (PARTITION BY customer_id) as total_spent
FROM sales
"""

window_df = spark.sql(window_sql)
window_df.explain()

# 3. Aggregate optimization
print("\n3. Aggregate Optimization:")
agg_sql = """
SELECT 
    region,
    category,
    COUNT(*) as transaction_count,
    SUM(amount) as total_amount,
    AVG(amount) as avg_amount,
    MIN(amount) as min_amount,
    MAX(amount) as max_amount
FROM sales
GROUP BY region, category
"""

agg_df = spark.sql(agg_sql)
agg_df.explain()

print("\nResults:")
agg_df.show()